# 03 · AutoML regression and scientific error metrics

**ML Science Lab · hands-on notebook**

**Learning goals:** predict a continuous outcome, interpret RMSE/MAE/R², compare to a mean baseline, understand metric selection.

Dataset: the educational scikit-learn Diabetes dataset (442 samples and 10 quantitative features). It models an outcome related to disease progression and is **not** for clinical use.

In [ ]:
%pip -q install "flaml[automl]>=2,<3"

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from flaml import AutoML
import numpy as np
import matplotlib.pyplot as plt

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=3, shuffle=True, random_state=42)
print('Development size:', len(X_dev), 'Test size:', len(X_test))

### 1. Compare a naive mean predictor and a random forest

The scoring rule `neg_root_mean_squared_error` returns a negative value because scikit-learn's model selection APIs maximize scores. Negate it to show positive RMSE: **smaller is better**.

In [ ]:
baselines = {
    'Predict mean': DummyRegressor(strategy='mean'),
    'Random forest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=2),
}
for name, estimator in baselines.items():
    scores = -cross_val_score(estimator, X_dev, y_dev,
                              cv=cv, scoring='neg_root_mean_squared_error')
    print(f'{name:18s} CV RMSE = {scores.mean():.2f} ± {scores.std():.2f}')

### 2. FLAML searches for low RMSE

Run once with **30 seconds**, or extend to 90 seconds if time permits. Search is restricted to development data.

In [ ]:
TIME_BUDGET = 30  # YOUR TURN: try 90 if you have time

automl = AutoML()
automl.fit(
    X_train=X_dev, y_train=y_dev,
    task='regression', metric='rmse', time_budget=TIME_BUDGET,
    eval_method='cv', n_splits=3,
    estimator_list=['rf', 'extra_tree', 'lgbm'],
    seed=42, n_jobs=2, verbose=0,
)
print('Best estimator:', automl.best_estimator)
print('Best configuration:', automl.best_config)
print('FLAML internal CV error (RMSE):', round(automl.best_loss, 2))

### 3. Final, once-only test evaluation

Only now show the test results for the preselected baseline and AutoML model.

In [ ]:
for name, estimator in [*baselines.items(), ('FLAML AutoML', automl)]:
    if name != 'FLAML AutoML':
        estimator.fit(X_dev, y_dev)
    prediction = estimator.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    print(f'{name:18s} test RMSE={rmse:.2f}, '
          f'MAE={mean_absolute_error(y_test, prediction):.2f}, '
          f'R²={r2_score(y_test, prediction):.3f}')

### Scientific discussion

- Why might you prefer MAE over RMSE if the measurements contain outliers?
- How would laboratory or participant clustering affect your validation strategy?
- How does error in the units of the target compare with the measurement precision?
- If you run many trials and select based on the **test set**, what goes wrong?
- Which additional checks would be needed before attempting any real biomedical application?